#### Environnement d'exécution :
MacBook Pro 2,6 GHz Intel Core i7 6 cœurs - 32 Go 2400 MHz DDR4

In [1]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
print(PROJECT_ROOT)

/Users/florianb/Downloads/ai-customer-insights-engine


In [2]:
import json
import time
from IPython.display import display, Markdown

from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.metrics.collections import (
    Faithfulness,
    AnswerRelevancy,
)
from ragas.embeddings import HuggingFaceEmbeddings
import pandas as pd
from markdown import markdown

from src.rag import retriever as retriever_module
from src.rag import rag_chain as rag_chain_module

from config import config

In [3]:
import importlib

importlib.reload(config)
importlib.reload(retriever_module)
importlib.reload(rag_chain_module)

<module 'src.rag.rag_chain' from '/Users/florianb/Downloads/ai-customer-insights-engine/src/rag/rag_chain.py'>

In [4]:
print(f"HUGGINGFACE_EMBEDDING_MODEL = {config.HUGGINGFACE_EMBEDDING_MODEL}")
print(f"INPUT_TOKEN_PRICE = {config.INPUT_TOKEN_PRICE}")
print(f"OUTPUT_TOKEN_PRICE = {config.OUTPUT_TOKEN_PRICE}")

HUGGINGFACE_EMBEDDING_MODEL = sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
INPUT_TOKEN_PRICE = 1.5e-07
OUTPUT_TOKEN_PRICE = 6e-07


In [5]:
evaluation_path = PROJECT_ROOT / "data/evaluation/evaluation_questions.json"

with open(evaluation_path, "r", encoding="utf-8") as f:
    questions = json.load(f)

In [6]:
questions

['Quels types de problèmes rencontrent les clients avec le service client ?',
 "Comment les clients décrivent-ils leur expérience lors de l'ouverture d'un compte ?",
 'Quelles difficultés les utilisateurs rencontrent-ils lors de la validation de leurs opérations ?',
 'Quels reproches sont faits concernant la réactivité du service client ?',
 'Comment les clients se sentent-ils face à des blocages de compte sans explication ?',
 'Quelles plaintes sont formulées concernant les frais bancaires ?',
 'Comment les clients perçoivent-ils la qualité des réponses fournies par le service client ?',
 'Quels problèmes sont signalés concernant la réception de cartes bancaires ?',
 'Comment les clients décrivent-ils leur expérience avec les délais de traitement des demandes ?',
 'Quelles frustrations sont exprimées concernant les communications par e-mail ?',
 'Quels avis négatifs sont partagés sur la gestion des incidents informatiques ?',
 'Comment les clients réagissent-ils face à des erreurs dan

In [7]:
# Retriever seul
retriever_only = retriever_module.create_retriever(
    model_name=config.HUGGINGFACE_EMBEDDING_MODEL,
    use_reranker=False,
    retriever_k=5,
)

# Retriever + reranker
retriever_reranker = retriever_module.create_retriever(
    model_name=config.HUGGINGFACE_EMBEDDING_MODEL,
    use_reranker=True,
    retriever_k=20,
    reranker_top_n=5,
)

In [8]:
# Chaîne RAG avec Retriever seul
rag_chain_retriever_only = rag_chain_module.create_rag_chain(
    llm_model="gpt-4o-mini",
    max_completion_tokens=500,
    retriever=retriever_only,
)

# Chaîne RAG avec Retriever + reranker
rag_chain_retriever_reranker = rag_chain_module.create_rag_chain(
    llm_model="gpt-4o-mini",
    max_completion_tokens=500,
    retriever=retriever_reranker,
)

### Faithfulness (Fidélité de la Réponse)

Le LLM-as-a-judge génère n affirmations à partir de la réponse générée, puis évalue pour chacune des n affirmations si elle est « retrouvée » dans l’ensemble des 5 contextes récupérées.

Formule : Nombre d’affirmations générées jugées fidèles par rapport aux contextes récupérés / Nombre total d’affirmations générées depuis la réponse générée

—> Score qui évalue la qualité de la génération du LLM (sa capacité à éviter les hallucinations).

In [9]:
client = AsyncOpenAI(
    api_key=config.OPENAI_API_KEY
)

evaluation_llm = llm_factory(
    client=client,
    model="gpt-4o-mini",
    max_tokens=1200,
    temperature=0,
)

scorer = Faithfulness(
    llm=evaluation_llm
)

In [10]:
# Résultats pour le retriever seul

# Chronomètre global
start_total = time.time()

results_retriever_only = []

for i, question in enumerate(questions, start=1):

    # Chronomètre du ainvoke()
    start_generation = time.time()

    result = await rag_chain_retriever_only.ainvoke(question)

    generation_duration = time.time() - start_generation

    answer = result["answer"].content

    retrieved_contexts = [
        doc.page_content
        for doc in result["context"]
    ]

    score = await scorer.ascore(
        user_input=question,
        response=answer,
        retrieved_contexts=retrieved_contexts,
    )

    input_tokens = result["answer"].response_metadata["token_usage"]["prompt_tokens"]
    output_tokens = result["answer"].response_metadata["token_usage"]["completion_tokens"]

    generation_cost = (
        input_tokens * config.INPUT_TOKEN_PRICE
        + output_tokens * config.OUTPUT_TOKEN_PRICE
    )
    
    results_retriever_only.append({
        "question_id": i,
        "question": question,
        "answer": answer,
        "retrieved_contexts": retrieved_contexts,
        "faithfulness": score.value,
        "generation_duration": generation_duration,
        "generation_cost": generation_cost,
    })

total_duration = time.time() - start_total
minutes, seconds = divmod(int(total_duration), 60)

print(f"Durée totale d'exécution : {minutes} min {seconds} s")

Durée totale d'exécution : 3 min 57 s


In [11]:
# Résultats pour le retriever + reranker

# Chronomètre global
start_total = time.time()

results_retriever_reranker = []

for i, question in enumerate(questions, start=1):

    # Chronomètre du ainvoke()
    start_generation = time.time()
    
    result = await rag_chain_retriever_reranker.ainvoke(question)

    generation_duration = time.time() - start_generation

    answer = result["answer"].content

    retrieved_contexts = [
        doc.page_content
        for doc in result["context"]
    ]

    score = await scorer.ascore(
        user_input=question,
        response=answer,
        retrieved_contexts=retrieved_contexts,
    )

    input_tokens = result["answer"].response_metadata["token_usage"]["prompt_tokens"]
    output_tokens = result["answer"].response_metadata["token_usage"]["completion_tokens"]

    generation_cost = (
        input_tokens * config.INPUT_TOKEN_PRICE
        + output_tokens * config.OUTPUT_TOKEN_PRICE
    )

    results_retriever_reranker.append({
        "question_id": i,
        "question": question,
        "answer": answer,
        "retrieved_contexts": retrieved_contexts,
        "faithfulness": score.value,
        "generation_duration": generation_duration,
        "generation_cost": generation_cost,
    })

total_duration = time.time() - start_total
minutes, seconds = divmod(int(total_duration), 60)

print(f"Durée totale d'exécution : {minutes} min {seconds} s")

Durée totale d'exécution : 7 min 10 s


In [ ]:
comparison = []

for i in range(len(questions)):
    score_retriever_only = results_retriever_only[i]["faithfulness"]
    score_retriever_reranker = results_retriever_reranker[i]["faithfulness"]

    duration_retriever_only = results_retriever_only[i]["generation_duration"]
    duration_retriever_reranker = results_retriever_reranker[i]["generation_duration"]

    cost_retriever_only = results_retriever_only[i]["generation_cost"]
    cost_retriever_reranker = results_retriever_reranker[i]["generation_cost"]

    comparison.append({
        "n°": i+1,
        "Question": questions[i],

        "Réponse Retriever seul": markdown(
            results_retriever_only[i]["answer"]
        ),
        "Contextes Retriever seul": "<br><br>".join(
            results_retriever_only[i]["retrieved_contexts"]
        ),
        "Score Retriever seul": score_retriever_only,
        "Durée Retriever seul": duration_retriever_only,
        "Coût Retriever seul": cost_retriever_only,

        "Réponse Retriever + Reranker": markdown(
            results_retriever_reranker[i]["answer"]
        ),
        "Contextes Retriever + Reranker": "<br><br>".join(
            results_retriever_reranker[i]["retrieved_contexts"]
        ),
        "Score Retriever + Reranker": score_retriever_reranker,
        "Durée Retriever + Reranker": duration_retriever_reranker,
        "Coût Retriever + Reranker": cost_retriever_reranker,

        "Écart des scores": score_retriever_reranker - score_retriever_only,
        "Écart des durées": duration_retriever_reranker - duration_retriever_only,
        "Écart des coûts": cost_retriever_reranker - cost_retriever_only,
    })

df_comparison = pd.DataFrame(comparison)

#df_comparison[["n°", "Question", "Réponse Retriever seul", "Score Retriever seul", "Réponse Retriever + Reranker", "Score Retriever + Reranker", "Écart"]].style.hide(axis="index")

In [26]:
# Moyennes des scores
mean_score_retriever_only = df_comparison["Score Retriever seul"].mean()
mean_score_retriever_reranker = df_comparison["Score Retriever + Reranker"].mean()

# Moyennes des durées
mean_duration_retriever_only = df_comparison["Durée Retriever seul"].mean()
mean_duration_retriever_reranker = df_comparison["Durée Retriever + Reranker"].mean()

# Moyennes des coûts
mean_cost_retriever_only = df_comparison["Coût Retriever seul"].mean()
mean_cost_retriever_reranker = df_comparison["Coût Retriever + Reranker"].mean()

df_comparison.loc[len(df_comparison)] = {
    "n°": "",
    "Question": "MOYENNE du Faithfulness (Fidélité de la Réponse) / de la Durée / du Coût",

    "Réponse Retriever seul": "",
    "Contextes Retriever seul": "",
    "Score Retriever seul": mean_score_retriever_only,
    "Durée Retriever seul": mean_duration_retriever_only,
    "Coût Retriever seul": mean_cost_retriever_only,

    "Réponse Retriever + Reranker": "",
    "Contextes Retriever + Reranker": "",
    "Score Retriever + Reranker": mean_score_retriever_reranker,
    "Durée Retriever + Reranker": mean_duration_retriever_reranker,
    "Coût Retriever + Reranker": mean_cost_retriever_reranker,

    "Écart des scores": mean_score_retriever_reranker - mean_score_retriever_only,
    "Écart des durées": mean_duration_retriever_reranker - mean_duration_retriever_only,
    "Écart des coûts": mean_cost_retriever_reranker - mean_cost_retriever_only,
}

#df_comparison[["n°", "Question", "Réponse Retriever seul", "Score Retriever seul", "Réponse Retriever + Reranker", "Score Retriever + Reranker", "Écart"]].style.hide(axis="index")

In [27]:
def color_score(row):

    styles = pd.Series("", index=row.index)

    score_retriever_only = row["Score Retriever seul"]
    score_retriever_reranker = row["Score Retriever + Reranker"]

    # Ligne "MOYENNE"
    if row["Question"] == "MOYENNE du Faithfulness (Fidélité de la Réponse) / de la Durée / du Coût":
        styles[:] = "font-weight: bold; border-top: 2px solid black;"

    # Comparaison des scores
    if score_retriever_only > score_retriever_reranker:
        styles["Score Retriever seul"] += " background-color: lightgreen;"
        styles["Score Retriever + Reranker"] += " background-color: lightcoral;"

    elif score_retriever_only < score_retriever_reranker:
        styles["Score Retriever seul"] += " background-color: lightcoral;"
        styles["Score Retriever + Reranker"] += " background-color: lightgreen;"

    return styles

In [28]:
display(Markdown("### Comparaison du Faithfulness (Fidélité de la Réponse)"))

df_comparison[
    [
        "n°",
        "Question",

        "Score Retriever seul",
        "Durée Retriever seul",
        "Coût Retriever seul",

        "Score Retriever + Reranker",
        "Durée Retriever + Reranker",
        "Coût Retriever + Reranker",        

        "Écart des scores",
        "Écart des durées",
        "Écart des coûts",
    ]
].style \
    .hide(axis="index") \
    .format({
        "Score Retriever seul": "{:.2f}",
        "Score Retriever + Reranker": "{:.2f}",
        "Durée Retriever seul": "{:.2f} s",
        "Durée Retriever + Reranker": "{:.2f} s",
        "Coût Retriever seul": "{:.6f} $",
        "Coût Retriever + Reranker": "{:.6f} $",
        "Écart des scores": "{:+.2f}",
        "Écart des durées": "{:+.2f} s",
        "Écart des coûts": "{:+.6f} $",
    }) \
    .apply(color_score, axis=1) \
    .set_properties(
        subset=[
            "Score Retriever seul",
            "Score Retriever + Reranker",
            "Écart des scores",
        ],
        **{"border-left": "1px solid black"}
    )

### Comparaison du Faithfulness (Fidélité de la Réponse)

n°,Question,Score Retriever seul,Durée Retriever seul,Coût Retriever seul,Score Retriever + Reranker,Durée Retriever + Reranker,Coût Retriever + Reranker,Écart des scores,Écart des durées,Écart des coûts
1,Quels types de problèmes rencontrent les clients avec le service client ?,1.00,6.40 s,0.000249 $,1.00,15.40 s,0.000319 $,+0.00,+9.00 s,+0.000070 $
2,Comment les clients décrivent-ils leur expérience lors de l'ouverture d'un compte ?,1.00,8.07 s,0.000303 $,1.00,10.44 s,0.000301 $,+0.00,+2.37 s,-0.000002 $
3,Quelles difficultés les utilisateurs rencontrent-ils lors de la validation de leurs opérations ?,1.00,2.46 s,0.000141 $,1.00,10.32 s,0.000246 $,+0.00,+7.86 s,+0.000106 $
4,Quels reproches sont faits concernant la réactivité du service client ?,1.00,3.49 s,0.000171 $,1.00,8.36 s,0.000165 $,+0.00,+4.86 s,-0.000006 $
5,Comment les clients se sentent-ils face à des blocages de compte sans explication ?,0.78,2.56 s,0.000193 $,1.00,11.83 s,0.000264 $,+0.22,+9.28 s,+0.000071 $
6,Quelles plaintes sont formulées concernant les frais bancaires ?,0.25,1.90 s,0.000135 $,0.43,10.24 s,0.000228 $,+0.18,+8.34 s,+0.000093 $
7,Comment les clients perçoivent-ils la qualité des réponses fournies par le service client ?,0.33,3.07 s,0.000206 $,0.67,6.79 s,0.000168 $,+0.33,+3.72 s,-0.000038 $
8,Quels problèmes sont signalés concernant la réception de cartes bancaires ?,1.00,2.05 s,0.000121 $,1.00,13.20 s,0.000323 $,+0.00,+11.15 s,+0.000202 $
9,Comment les clients décrivent-ils leur expérience avec les délais de traitement des demandes ?,0.22,2.75 s,0.000184 $,0.80,10.75 s,0.000294 $,+0.58,+8.00 s,+0.000110 $
10,Quelles frustrations sont exprimées concernant les communications par e-mail ?,1.00,4.30 s,0.000263 $,1.00,13.05 s,0.000270 $,+0.00,+8.75 s,+0.000007 $


In [29]:
pd.set_option("display.max_colwidth", None)

In [30]:
display(Markdown("### Comparaison du Faithfulness (Fidélité de la Réponse)"))

df_comparison.style \
    .hide(axis="index") \
    .format({
        "Score Retriever seul": "{:.2f}",
        "Score Retriever + Reranker": "{:.2f}",
        "Durée Retriever seul": "{:.2f} s",
        "Durée Retriever + Reranker": "{:.2f} s",
        "Coût Retriever seul": "{:.6f} $",
        "Coût Retriever + Reranker": "{:.6f} $",
        "Écart des scores": "{:+.2f}",
        "Écart des durées": "{:+.2f} s",
        "Écart des coûts": "{:+.6f} $",
    }) \
    .apply(color_score, axis=1) \
    .set_properties(
        subset=[
            "Réponse Retriever seul",
            "Réponse Retriever + Reranker",
            "Écart des scores",
        ],
        **{"border-left": "1px solid black"}
    )

### Comparaison du Faithfulness (Fidélité de la Réponse)

n°,Question,Réponse Retriever seul,Contextes Retriever seul,Score Retriever seul,Durée Retriever seul,Coût Retriever seul,Réponse Retriever + Reranker,Contextes Retriever + Reranker,Score Retriever + Reranker,Durée Retriever + Reranker,Coût Retriever + Reranker,Écart des scores,Écart des durées,Écart des coûts
1,Quels types de problèmes rencontrent les clients avec le service client ?,"Les clients rencontrent plusieurs types de problèmes avec le service client, notamment : Incompétence : Plusieurs avis mentionnent un service client incompétent, ce qui suggère un manque de compétence des conseillers. Processus incompréhensibles : Un client a signalé que les processus étaient difficiles à comprendre, rendant même des actions simples comme la fermeture de compte compliquées. Raccrochage au nez : Un autre avis indique que le service client a raccroché au nez du client, ce qui montre un manque de professionnalisme. Communication d’information partielle : Un client a noté que les informations fournies étaient incomplètes et parfois intentionnellement trompeuses. Réponses contradictoires : Les clients ont également signalé des réponses contradictoires selon le conseiller contacté, ce qui crée de la confusion. Absence de réponse : Certains clients ont fait état d'une absence de réponse à leurs messages, ce qui est un problème de réactivité. Ces problèmes contribuent à une expérience client négative.","Un service client incompétent - des processus incompréhensibles - même fermer son compte est compliquéservice client incompétant, raccroche au nezRéactivité du service clientService clients médiocre : incompétence, communication d’information partielle de manière intentionnelle, réponses contradictoires en fonction du conseiller en ligne, absence de réponse à vos messages…Un service client très compliqué et vraiment pas sympa",1.00,6.40 s,0.000249 $,"Les clients rencontrent plusieurs types de problèmes avec le service client, notamment : Mauvaise communication : Les clients signalent que le suivi des dossiers est mal géré, avec des informations qui passent de main en main sans qu'un service dédié prenne les choses au sérieux. De plus, il y a des difficultés à contacter le service client par téléphone ou par e-mail, et parfois les e-mails restent sans réponse. Difficulté d'accès aux systèmes de réclamation : Certains clients trouvent que le système de réclamation est difficile d'accès, limité et ne conserve pas l'historique des dossiers, ce qui complique la communication. Incompétence et incohérence : Des avis mentionnent une incompétence du service client, avec des réponses contradictoires selon le conseiller contacté et une communication d'informations partielles. Absence de réponse : Il y a des cas où les clients ne reçoivent pas de réponse à leurs messages, ce qui aggrave leur frustration. Attitude des conseillers : Certains clients décrivent les conseillers comme étant froids et peu courtois lorsqu'ils parviennent à les contacter. Service client inexistant : Dans certains cas, les clients estiment que le service client est inexistant, surtout lors de problèmes nécessitant une assistance. Ces problèmes contribuent à une expérience client très insatisfaisante.","Le service client est déplorable, le suivie d'un dossier sensible passe de main en main, sans qu'un service dédié prenne les choses aux sérieux afin de régler le problèmes. De plus ils refusent de vous communiqué autrement que par leur système de réclamation qui est non seulement difficile d'accès, fortement limité mais n'enregistre pas l'historique de votre dossier, ce qui complique encore la communication. Je déconseille fortement car au moindre soucis ça devient l'enfer.Le service client est pourri:Très bonne Banque, leur application est fluide, clair, puis simple à utiliserLe seul problème, c’est le service client c’est le point négatif. C’est le point noir très difficile de les contacter par téléphone ou par mail. Parfois il ne répond même pas aux e-mails. Et quand tu as l

### Answer Relevancy (Pertinence de la Réponse)

Le LLM-as-a-judge évalue si la réponse générée est pertinente vis-à-vis de la question en générant des questions à partir de la réponse générée qui sont ensuite comparées sémantiquement à la question (d'où le paramètre "embeddings").

—> Score qui évalue la qualité de la génération du LLM.

In [31]:
client = AsyncOpenAI(
    api_key=config.OPENAI_API_KEY
)

evaluation_llm = llm_factory(
    client=client,
    model="gpt-4o-mini",
    max_tokens=1200,
    temperature=0,
)

embedding_function = HuggingFaceEmbeddings(
    model="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
)

scorer = AnswerRelevancy(
    llm=evaluation_llm,
    embeddings=embedding_function
)

In [32]:
# Résultats pour le retriever seul

# Chronomètre global
start_total = time.time()

results_retriever_only = []

for i, question in enumerate(questions, start=1):

    # Chronomètre du ainvoke()
    start_generation = time.time()

    result = await rag_chain_retriever_only.ainvoke(question)

    generation_duration = time.time() - start_generation

    answer = result["answer"].content

    score = await scorer.ascore(
        user_input=question,
        response=answer,
    )

    input_tokens = result["answer"].response_metadata["token_usage"]["prompt_tokens"]
    output_tokens = result["answer"].response_metadata["token_usage"]["completion_tokens"]

    generation_cost = (
        input_tokens * config.INPUT_TOKEN_PRICE
        + output_tokens * config.OUTPUT_TOKEN_PRICE
    )

    results_retriever_only.append({
        "question_id": i,
        "question": question,
        "answer": answer,
        "answer_relevancy": score.value,
        "generation_duration": generation_duration,
        "generation_cost": generation_cost,
    })

total_duration = time.time() - start_total
minutes, seconds = divmod(int(total_duration), 60)

print(f"Durée totale d'exécution : {minutes} min {seconds} s")

Durée totale d'exécution : 2 min 32 s


In [33]:
# Résultats pour le retriever + reranker

# Chronomètre global
start_total = time.time()

results_retriever_reranker = []

for i, question in enumerate(questions, start=1):

    # Chronomètre du ainvoke()
    start_generation = time.time()
    
    result = await rag_chain_retriever_reranker.ainvoke(question)

    generation_duration = time.time() - start_generation
    
    answer = result["answer"].content

    score = await scorer.ascore(
        user_input=question,
        response=answer,
    )

    input_tokens = result["answer"].response_metadata["token_usage"]["prompt_tokens"]
    output_tokens = result["answer"].response_metadata["token_usage"]["completion_tokens"]

    generation_cost = (
        input_tokens * config.INPUT_TOKEN_PRICE
        + output_tokens * config.OUTPUT_TOKEN_PRICE
    )

    results_retriever_reranker.append({
        "question_id": i,
        "question": question,
        "answer": answer,
        "answer_relevancy": score.value,
        "generation_duration": generation_duration,
        "generation_cost": generation_cost,
    })

total_duration = time.time() - start_total
minutes, seconds = divmod(int(total_duration), 60)

print(f"Durée totale d'exécution : {minutes} min {seconds} s")

Durée totale d'exécution : 8 min 5 s


In [44]:
comparison = []

for i in range(len(questions)):
    score_retriever_only = results_retriever_only[i]["answer_relevancy"]
    score_retriever_reranker = results_retriever_reranker[i]["answer_relevancy"]

    duration_retriever_only = results_retriever_only[i]["generation_duration"]
    duration_retriever_reranker = results_retriever_reranker[i]["generation_duration"]

    cost_retriever_only = results_retriever_only[i]["generation_cost"]
    cost_retriever_reranker = results_retriever_reranker[i]["generation_cost"]

    comparison.append({
        "n°": i+1,
        "Question": questions[i],

        "Réponse Retriever seul": markdown(
            results_retriever_only[i]["answer"]
        ),
        "Score Retriever seul": score_retriever_only,
        "Durée Retriever seul": duration_retriever_only,
        "Coût Retriever seul": cost_retriever_only,

        "Réponse Retriever + Reranker": markdown(
            results_retriever_reranker[i]["answer"]
        ),
        "Score Retriever + Reranker": score_retriever_reranker,
        "Durée Retriever + Reranker": duration_retriever_reranker,
        "Coût Retriever + Reranker": cost_retriever_reranker,

        "Écart des scores": score_retriever_reranker - score_retriever_only,
        "Écart des durées": duration_retriever_reranker - duration_retriever_only,
        "Écart des coûts": cost_retriever_reranker - cost_retriever_only,
    })

df_comparison = pd.DataFrame(comparison)

#df_comparison.style.hide(axis="index")

In [45]:
# Moyennes des scores
mean_score_retriever_only = df_comparison["Score Retriever seul"].mean()
mean_score_retriever_reranker = df_comparison["Score Retriever + Reranker"].mean()

# Moyennes des durées
mean_duration_retriever_only = df_comparison["Durée Retriever seul"].mean()
mean_duration_retriever_reranker = df_comparison["Durée Retriever + Reranker"].mean()

# Moyennes des coûts
mean_cost_retriever_only = df_comparison["Coût Retriever seul"].mean()
mean_cost_retriever_reranker = df_comparison["Coût Retriever + Reranker"].mean()

df_comparison.loc[len(df_comparison)] = {
    "n°": "",
    "Question": "MOYENNE du Answer Relevancy (Pertinence de la Réponse) / de la Durée / du Coût",

    "Réponse Retriever seul": "",
    "Score Retriever seul": mean_score_retriever_only,
    "Durée Retriever seul": mean_duration_retriever_only,
    "Coût Retriever seul": mean_cost_retriever_only,

    "Réponse Retriever + Reranker": "",
    "Score Retriever + Reranker": mean_score_retriever_reranker,
    "Durée Retriever + Reranker": mean_duration_retriever_reranker,
    "Coût Retriever + Reranker": mean_cost_retriever_reranker,

    "Écart des scores": mean_score_retriever_reranker - mean_score_retriever_only,
    "Écart des durées": mean_duration_retriever_reranker - mean_duration_retriever_only,
    "Écart des coûts": mean_cost_retriever_reranker - mean_cost_retriever_only,
}

#df_comparison.style.hide(axis="index")

In [46]:
def color_score(row):

    styles = pd.Series("", index=row.index)

    score_retriever_only = row["Score Retriever seul"]
    score_retriever_reranker = row["Score Retriever + Reranker"]

    # Ligne "MOYENNE"
    if row["Question"] == "MOYENNE du Answer Relevancy (Pertinence de la Réponse) / de la Durée / du Coût":
        styles[:] = "font-weight: bold; border-top: 2px solid black;"

    # Comparaison des scores
    if score_retriever_only > score_retriever_reranker:
        styles["Score Retriever seul"] += " background-color: lightgreen;"
        styles["Score Retriever + Reranker"] += " background-color: lightcoral;"

    elif score_retriever_only < score_retriever_reranker:
        styles["Score Retriever seul"] += " background-color: lightcoral;"
        styles["Score Retriever + Reranker"] += " background-color: lightgreen;"

    return styles

In [47]:
display(Markdown("### Comparaison du Answer Relevancy (Pertinence de la Réponse)"))

df_comparison[
    [
        "n°",
        "Question",

        "Score Retriever seul",
        "Durée Retriever seul",
        "Coût Retriever seul",

        "Score Retriever + Reranker",
        "Durée Retriever + Reranker",
        "Coût Retriever + Reranker",

        "Écart des scores",
        "Écart des durées",
        "Écart des coûts",
    ]
].style \
    .hide(axis="index") \
    .format({
        "Score Retriever seul": "{:.2f}",
        "Score Retriever + Reranker": "{:.2f}",
        "Durée Retriever seul": "{:.2f} s",
        "Durée Retriever + Reranker": "{:.2f} s",
        "Coût Retriever seul": "{:.6f} $",
        "Coût Retriever + Reranker": "{:.6f} $",
        "Écart des scores": "{:+.2f}",
        "Écart des durées": "{:+.2f} s",
        "Écart des coûts": "{:+.6f} $",
    }) \
    .apply(color_score, axis=1) \
    .set_properties(
        subset=[
            "Score Retriever seul",
            "Score Retriever + Reranker",
            "Écart des scores",
        ],
        **{"border-left": "1px solid black"}
    )

### Comparaison du Answer Relevancy (Pertinence de la Réponse)

n°,Question,Score Retriever seul,Durée Retriever seul,Coût Retriever seul,Score Retriever + Reranker,Durée Retriever + Reranker,Coût Retriever + Reranker,Écart des scores,Écart des durées,Écart des coûts
1,Quels types de problèmes rencontrent les clients avec le service client ?,1.00,4.81 s,0.000252 $,1.00,19.99 s,0.000317 $,+0.00,+15.17 s,+0.000065 $
2,Comment les clients décrivent-ils leur expérience lors de l'ouverture d'un compte ?,1.00,4.02 s,0.000279 $,1.00,20.84 s,0.000300 $,+0.00,+16.83 s,+0.000021 $
3,Quelles difficultés les utilisateurs rencontrent-ils lors de la validation de leurs opérations ?,1.00,1.32 s,0.000141 $,0.99,29.37 s,0.000246 $,-0.01,+28.05 s,+0.000105 $
4,Quels reproches sont faits concernant la réactivité du service client ?,0.99,2.13 s,0.000191 $,0.99,19.60 s,0.000160 $,+0.00,+17.47 s,-0.000031 $
5,Comment les clients se sentent-ils face à des blocages de compte sans explication ?,1.00,1.36 s,0.000204 $,1.00,20.65 s,0.000254 $,+0.00,+19.29 s,+0.000050 $
6,Quelles plaintes sont formulées concernant les frais bancaires ?,0.00,1.23 s,0.000134 $,0.86,19.17 s,0.000228 $,+0.86,+17.94 s,+0.000093 $
7,Comment les clients perçoivent-ils la qualité des réponses fournies par le service client ?,1.00,2.03 s,0.000182 $,1.00,13.07 s,0.000168 $,+0.00,+11.03 s,-0.000014 $
8,Quels problèmes sont signalés concernant la réception de cartes bancaires ?,0.99,1.32 s,0.000121 $,0.88,28.18 s,0.000323 $,-0.11,+26.86 s,+0.000202 $
9,Comment les clients décrivent-ils leur expérience avec les délais de traitement des demandes ?,0.86,2.22 s,0.000190 $,0.87,23.66 s,0.000291 $,+0.01,+21.43 s,+0.000101 $
10,Quelles frustrations sont exprimées concernant les communications par e-mail ?,1.00,3.61 s,0.000234 $,0.98,30.50 s,0.000269 $,-0.02,+26.89 s,+0.000035 $


In [48]:
pd.set_option("display.max_colwidth", None)

In [49]:
display(Markdown("### Comparaison du Answer Relevancy (Pertinence de la Réponse)"))

df_comparison.style \
    .hide(axis="index") \
    .format({
        "Score Retriever seul": "{:.2f}",
        "Score Retriever + Reranker": "{:.2f}",
        "Durée Retriever seul": "{:.2f} s",
        "Durée Retriever + Reranker": "{:.2f} s",
        "Coût Retriever seul": "{:.6f} $",
        "Coût Retriever + Reranker": "{:.6f} $",
        "Écart des scores": "{:+.2f}",
        "Écart des durées": "{:+.2f} s",
        "Écart des coûts": "{:+.6f} $",
    }) \
    .apply(color_score, axis=1) \
    .set_properties(
        subset=[
            "Réponse Retriever seul",
            "Réponse Retriever + Reranker",
            "Écart des scores",
        ],
        **{"border-left": "1px solid black"}
    )

### Comparaison du Answer Relevancy (Pertinence de la Réponse)

n°,Question,Réponse Retriever seul,Score Retriever seul,Durée Retriever seul,Coût Retriever seul,Réponse Retriever + Reranker,Score Retriever + Reranker,Durée Retriever + Reranker,Coût Retriever + Reranker,Écart des scores,Écart des durées,Écart des coûts
1,Quels types de problèmes rencontrent les clients avec le service client ?,"Les clients rencontrent plusieurs types de problèmes avec le service client, notamment : Incompétence : Plusieurs avis mentionnent un service client incompétent, ce qui suggère un manque de compétence des conseillers. Processus incompréhensibles : Les clients trouvent que les processus sont difficiles à comprendre, rendant les interactions frustrantes. Difficulté à fermer un compte : Un client a signalé que même la fermeture de son compte était compliquée. Raccrochage au nez : Un autre avis indique que le service client a raccroché au nez du client, ce qui montre un manque de professionnalisme. Communication d’information partielle : Il y a des allégations de communication d'informations incomplètes ou intentionnellement partielles. Réponses contradictoires : Les clients reçoivent des réponses différentes selon le conseiller, ce qui crée de la confusion. Absence de réponse : Certains clients se plaignent de ne pas recevoir de réponse à leurs messages. Ces problèmes contribuent à une expérience client négative.",1.00,4.81 s,0.000252 $,"Les clients rencontrent plusieurs types de problèmes avec le service client, notamment : Mauvaise communication : Les clients signalent que le suivi des dossiers est mal géré, avec des informations qui passent de main en main sans qu'un service dédié prenne les choses au sérieux. De plus, il y a des difficultés à contacter le service client par téléphone ou par email, et parfois les emails restent sans réponse. Difficulté d'accès au système de réclamation : Le système de réclamation est décrit comme difficile d'accès et limité, sans enregistrement de l'historique des dossiers, ce qui complique la communication. Incompétence et incohérence : Certains clients mentionnent une incompétence du service client, avec des informations partielles communiquées de manière intentionnelle et des réponses contradictoires selon le conseiller. Absence de réponse : Il y a des cas où les clients ne reçoivent pas de réponse à leurs messages. Attitude des conseillers : Les clients rapportent que les conseillers sont parfois froids et peu courtois lorsqu'ils parviennent à les contacter. Service client inexistant : Dans certains cas, les clients estiment qu'il est impossible de parler à quelqu'un qui ait un libre arbitre pour résoudre leurs problèmes. Ces problèmes contribuent à une expérience client très insatisfaisante.",1.00,19.99 s,0.000317 $,+0.00,+15.17 s,+0.000065 $
2,Comment les clients décrivent-ils leur expérience lors de l'ouverture d'un compte ?,"Les clients décrivent leur expérience lors de l'ouverture d'un compte de manière variée : Insatisfaction : Un client a exprimé une grande insatisfaction concernant le service client lors de l'ouverture d'un compte pour sa mère, mentionnant un manque de suivi après un rendez-vous fixé pour un rappel. Satisfaction élevée : À l'inverse, un autre client a eu une ""super expérience"" d'ouverture de compte chez Hellobank, soulignant la clarté des démarches, le suivi et la réactivité des conseillers, ce qui lui a donné confiance pour la suite. Expérience mitigée : Un client de Monabanq a trouvé que l'ouverture d'un compte était bien encadrée avec des conseillers avenants, mais a rencontré des difficultés pour obtenir un suivi après l'ouverture, notamment en ce qui concerne l'attente de sa carte bleue. En résumé, les expériences varient de l'insatisfaction à une satisfaction totale, avec des points positifs sur la clarté et le suivi, mais aussi des critiques sur le manque de réactivité après l'ouverture du compte.",1.00,4.02 s,0.000279 $,"Les clients décrivent leur expérience lors de l'ouverture d'un compte de manière variée : Expérience positive